In [2]:
# ============================================================
# Import Libraries
# ============================================================

import joblib
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [3]:
# ============================================================
# Load Existing XGBoost Model
# ============================================================

xgb_saved = joblib.load(
    "../models/higgs_xgboost.pkl"
)

print(xgb_saved)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)


In [4]:
# ============================================================
# Load Processed Dataset
# ============================================================

from preprocessing_22 import load_data_unscaled

(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
    feature_names
) = load_data_unscaled()

In [5]:
# ============================================================
# Evaluate Saved Model
# ============================================================

y_prob = xgb_saved.predict_proba(X_test)[:,1]

baseline_auc = roc_auc_score(
    y_test,
    y_prob
)

print(
    "Baseline ROC-AUC:",
    round(baseline_auc, 4)
)

Baseline ROC-AUC: 0.9105


In [6]:
# ============================================================
# Hyperparameter Search Space
# ============================================================

param_grid = {

    # Tree complexity
    "max_depth": [5, 6, 7, 8],

    # Number of trees
    "n_estimators": [1500, 2000, 2500, 3000],

    # Learning rate
    "learning_rate": [0.01, 0.02, 0.03, 0.05],

    # Row sampling
    "subsample": [0.7, 0.8, 0.9],

    # Feature sampling
    "colsample_bytree": [0.7, 0.8, 0.9],

    # Regularization
    "reg_alpha": [0, 0.05, 0.1, 0.2],

    "reg_lambda": [1, 2, 3, 5],

    # Child weight
    "min_child_weight": [1, 3, 5, 7],

    # Split regularization
    "gamma": [0, 0.05, 0.1, 0.2]
}

In [7]:
# ============================================================
# Base XGBoost Model
# ============================================================

from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",

    scale_pos_weight=1.93,

    random_state=42,
    n_jobs=-1
)

In [8]:
# ============================================================
# Hyperparameter Tuning
# ============================================================

from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    estimator=xgb,

    param_distributions=param_grid,

    n_iter=50,              # increase search

    scoring="roc_auc",

    cv=3,

    verbose=2,

    random_state=42,

    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Best Parameters
# ============================================================

print(
    search.best_params_
)

print(
    search.best_score_
)

{'subsample': 0.8, 'n_estimators': 1000, 'min_child_weight': 5, 'max_depth': 8, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 0.7}
0.9139471812863004


In [ ]:
# ============================================================
# Evaluate Tuned Model
# ============================================================

best_xgb = search.best_estimator_

y_pred = best_xgb.predict(X_test)

y_prob = best_xgb.predict_proba(X_test)[:, 1]

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_prob):.4f}")

Accuracy: 0.8431
Precision: 0.7906
Recall: 0.7355
F1: 0.7621
ROC-AUC: 0.9119


In [ ]:
"""# ============================================================
# Save Tuned Model
# ============================================================

joblib.dump(
    best_xgb,
    "../models/higgs_xgboost_tuned.pkl"
)

print(
    "Tuned model saved successfully."
)"""

Tuned model saved successfully.


## Check Overfitting 

In [ ]:
from sklearn.metrics import roc_auc_score

train_prob = best_xgb.predict_proba(X_train)[:,1]
val_prob = best_xgb.predict_proba(X_val)[:,1]
test_prob = best_xgb.predict_proba(X_test)[:,1]

print(
    "Train AUC:",
    roc_auc_score(y_train, train_prob)
)

print(
    "Validation AUC:",
    roc_auc_score(y_val, val_prob)
)

print(
    "Test AUC:",
    roc_auc_score(y_test, test_prob)
)

Train AUC: 0.9374349054922487
Validation AUC: 0.9152520155860072
Test AUC: 0.9118815767587398
